## Tips

- **Start simple** — a single GRU with 64 units is a good first attempt.
  Only add complexity (stacking, dropout, bidirectional) if the simple model
  is clearly underfitting or overfitting.

- **Use many epochs** — RNNs on time series converge slowly. 10 or 20 epochs
  is almost never enough. Use **at least 100 epochs** and let early stopping
  decide when to stop. The best checkpoint is saved automatically by
  `ModelCheckpoint` — you will not miss the optimal point even if you
  train for too long.

- **Watch the train/val gap** — if train MAE is much lower than val MAE
  you are overfitting. Add dropout, reduce `hidden_size`, or increase
  early stopping patience to give regularization more time to work.

- **Use TensorBoard** — run `tensorboard --logdir runs` in your terminal
  to monitor train and val curves in real time. A healthy training curve
  shows both losses decreasing together. A diverging val curve means overfitting.

- **Learning rate matters** — if the loss is not decreasing after the first
  few epochs, try a lower learning rate (`1e-4` instead of `1e-3`).
  Use `ReduceLROnPlateau` to automatically decay the learning rate when
  val MAE stops improving.

- **The sklearn baseline is hard to beat** — `HistGradientBoosting` with
  explicit lag features is a very strong baseline for tabular time series.
  This is not a failure of the RNN — it reflects a fundamental difference
  between the two approaches: tree models get temporal information from
  hand-crafted features, while RNNs must learn it from raw sequences.
  Getting within 10 bikes/hour of the sklearn baseline (< 44 bikes/hour)
  is an excellent result.

In [4]:
import os
import numpy as np
import joblib

from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [5]:
save_dir = "data/bike_processed"

# --- Load arrays ---
raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
counts = np.load(os.path.join(save_dir, "counts.npy"))
count_stats = np.load(os.path.join(save_dir, "count_stats.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
naive_mae   = np.load(os.path.join(save_dir, "naive_mae.npy"))
preprocessor = joblib.load(os.path.join(save_dir, "preprocessor_rnn.pkl"))

# --- Recover split sizes ---
num_train = len(train_idx)
num_val   = len(val_idx)
num_test  = len(test_idx)

# --- Count stats in the training set ---
count_mean  = count_stats[0]
count_std   = count_stats[1]

# --- Sanity check ---
print(f"raw_data shape:    {raw_data.shape}")
print(f"counts shape: {counts.shape}")
print(f"train/val/test:    {num_train} / {num_val} / {num_test}")
print(f"counts range: {counts.min():.0f} to {counts.max():.0f} bikes/hour")
print(f"\nNaive baseline — Val MAE: {naive_mae[0]:.2f} | Test MAE: {naive_mae[1]:.2f} bikes/hour")

raw_data shape:    (17210, 28)
counts shape: (17210,)
train/val/test:    12047 / 2581 / 2582
counts range: 1 to 977 bikes/hour

Naive baseline — Val MAE: 93.26 | Test MAE: 80.78 bikes/hour


In [6]:
count_norm = (counts - count_mean) / count_std

In [7]:
class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

In [8]:
sampling_rate   = 1    # already hourly
sequence_length = 24   # look back 24 hours
delay           = 1    # predict 1 hour ahead
batch_size      = 256

train_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train - sequence_length - delay + 1,
)
val_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train, end_index=num_train + num_val - sequence_length - delay + 1,
)
test_dataset = TimeseriesDataset(
    data=raw_data, targets=count_norm,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train + num_val, end_index=len(raw_data) - sequence_length - delay + 1,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# --- Sanity check ---
inputs, targets = next(iter(train_loader))
print(f"Input shape:  {inputs.shape}")   # (256, 24, num_features)
print(f"Target shape: {targets.shape}")  # (256,)
print(f"Target range: {targets.min():.0f} to {targets.max():.0f}")

Input shape:  torch.Size([256, 24, 28])
Target shape: torch.Size([256])
Target range: -1 to 4


## Models

In [9]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae  = 0.0
    n          = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss  = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += inputs.size(0)
    return total_loss / n, total_mae / n  # MAE in normalized units


def get_predictions(model, dataset):
    model.eval()
    all_preds   = []
    all_targets = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            all_preds.append(preds * count_std + count_mean)  # un-normalize to bikes/hour
            all_targets.append(targets.numpy() * count_std + count_mean)  # un-normalize targets
    return np.concatenate(all_preds), np.concatenate(all_targets)

In [10]:
class ModelCheckpoint:
    """Saves the best model based on a monitored metric."""
    def __init__(self, filepath, monitor="val_mae", mode="min", verbose=True):
        self.filepath = filepath
        self.monitor  = monitor
        self.verbose  = verbose
        self.best     = float("inf") if mode == "min" else float("-inf")
        self.mode     = mode

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Best model saved ({self.monitor}: {value * count_std:.2f} bikes/hour)")
        return improved


class EarlyStopping:
    """Stops training when a monitored metric stops improving."""
    def __init__(self, monitor="val_mae", patience=10, min_delta=1e-4, mode="min"):
        self.monitor     = monitor
        self.patience    = patience
        self.min_delta   = min_delta
        self.mode        = mode
        self.best        = float("inf") if mode == "min" else float("-inf")
        self.counter     = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  Early stopping triggered (no improvement for {self.patience} epochs)")
        return improved


class ReduceLROnPlateau:
    """Wraps PyTorch scheduler with the same callback interface."""
    def __init__(self, optimizer, monitor="val_mae", patience=3, factor=0.5):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

### Simple RNN

In [11]:
class SimpleRNNModel(nn.Module):
    def __init__(self, num_features, hidden_size=64):
        super().__init__()
        self.rnn  = nn.RNN(input_size=num_features, hidden_size=hidden_size,
                           batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)


In [12]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = SimpleRNNModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/bike_simple_rnn")

callbacks = [
    ModelCheckpoint("bike_simple_rnn_best.pt", monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=20),
    ReduceLROnPlateau(optimizer, monitor="val_mae", patience=5),
]

# --- Training loop ---
epochs = 200
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    # Un-normalize MAE back to bikes/hour
    train_mae_bikes = train_mae * count_std
    val_mae_bikes   = val_mae   * count_std

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae_bikes:.2f} bikes/hour | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae_bikes:.2f} bikes/hour")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

    if any(isinstance(cb, EarlyStopping) and cb.should_stop for cb in callbacks):
        break

writer.close()

Epoch 01 — train loss: 0.6032, train MAE: 87.32 bikes/hour | val loss: 1.2788, val MAE: 134.22 bikes/hour
  ✓ Best model saved (val_mae: 134.22 bikes/hour)
Epoch 02 — train loss: 0.3664, train MAE: 67.05 bikes/hour | val loss: 1.0760, val MAE: 123.85 bikes/hour
  ✓ Best model saved (val_mae: 123.85 bikes/hour)
Epoch 03 — train loss: 0.2848, train MAE: 58.17 bikes/hour | val loss: 0.7493, val MAE: 95.55 bikes/hour
  ✓ Best model saved (val_mae: 95.55 bikes/hour)
Epoch 04 — train loss: 0.2071, train MAE: 49.39 bikes/hour | val loss: 0.5864, val MAE: 81.05 bikes/hour
  ✓ Best model saved (val_mae: 81.05 bikes/hour)
Epoch 05 — train loss: 0.1617, train MAE: 43.74 bikes/hour | val loss: 0.4968, val MAE: 75.23 bikes/hour
  ✓ Best model saved (val_mae: 75.23 bikes/hour)
Epoch 06 — train loss: 0.1457, train MAE: 41.19 bikes/hour | val loss: 0.4704, val MAE: 75.89 bikes/hour
Epoch 07 — train loss: 0.1272, train MAE: 38.49 bikes/hour | val loss: 0.3914, val MAE: 67.69 bikes/hour
  ✓ Best model s

In [13]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("bike_simple_rnn_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae * count_std:.2f} bikes/hour")


Test MAE: 40.56 bikes/hour


### GRU Model

In [28]:
class GRUModel(nn.Module):
    def __init__(self, num_features, hidden_size=64):
        super().__init__()
        self.lstm   = nn.GRU(input_size=num_features, hidden_size=hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :]).squeeze(-1)

In [29]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = GRUModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/bike_gru")

callbacks = [
    ModelCheckpoint("bike_gru_best.pt", monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=30),
    ReduceLROnPlateau(optimizer, monitor="val_mae", patience=5),
]

# --- Training loop ---
epochs = 200
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    train_mae_bikes = train_mae * count_std
    val_mae_bikes   = val_mae   * count_std

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae_bikes:.2f} bikes/hour | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae_bikes:.2f} bikes/hour")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

    if any(isinstance(cb, EarlyStopping) and cb.should_stop for cb in callbacks):
        break

writer.close()

Epoch 01 — train loss: 0.6669, train MAE: 93.62 bikes/hour | val loss: 1.2465, val MAE: 136.17 bikes/hour
  ✓ Best model saved (val_mae: 136.17 bikes/hour)
Epoch 02 — train loss: 0.4012, train MAE: 67.91 bikes/hour | val loss: 0.9760, val MAE: 113.28 bikes/hour
  ✓ Best model saved (val_mae: 113.28 bikes/hour)
Epoch 03 — train loss: 0.3509, train MAE: 64.61 bikes/hour | val loss: 0.9066, val MAE: 108.93 bikes/hour
  ✓ Best model saved (val_mae: 108.93 bikes/hour)
Epoch 04 — train loss: 0.3127, train MAE: 60.49 bikes/hour | val loss: 0.7862, val MAE: 96.84 bikes/hour
  ✓ Best model saved (val_mae: 96.84 bikes/hour)
Epoch 05 — train loss: 0.2492, train MAE: 52.73 bikes/hour | val loss: 0.5767, val MAE: 85.11 bikes/hour
  ✓ Best model saved (val_mae: 85.11 bikes/hour)
Epoch 06 — train loss: 0.1779, train MAE: 43.86 bikes/hour | val loss: 0.4131, val MAE: 68.63 bikes/hour
  ✓ Best model saved (val_mae: 68.63 bikes/hour)
Epoch 07 — train loss: 0.1228, train MAE: 36.58 bikes/hour | val loss:

In [30]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("bike_gru_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae * count_std:.2f} bikes/hour")


Test MAE: 36.96 bikes/hour


### Simple LSTM

In [46]:
class LSTMModel(nn.Module):
    def __init__(self, num_features, hidden_size=64):
        super().__init__()
        self.lstm   = nn.LSTM(input_size=num_features, hidden_size=hidden_size, batch_first=True)
        self.linear = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :]).squeeze(-1)

In [49]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = LSTMModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/bike_lstm")

callbacks = [
    ModelCheckpoint("bike_lstm_best.pt", monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=30),
    ReduceLROnPlateau(optimizer, monitor="val_mae", patience=3),
]

# --- Training loop ---
epochs = 200
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    train_mae_bikes = train_mae * count_std
    val_mae_bikes   = val_mae   * count_std

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae_bikes:.2f} bikes/hour | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae_bikes:.2f} bikes/hour")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

    if any(isinstance(cb, EarlyStopping) and cb.should_stop for cb in callbacks):
        break

writer.close()

Epoch 01 — train loss: 0.6504, train MAE: 92.47 bikes/hour | val loss: 1.1232, val MAE: 117.27 bikes/hour
  ✓ Best model saved (val_mae: 117.27 bikes/hour)
Epoch 02 — train loss: 0.3625, train MAE: 65.21 bikes/hour | val loss: 0.9287, val MAE: 103.02 bikes/hour
  ✓ Best model saved (val_mae: 103.02 bikes/hour)
Epoch 03 — train loss: 0.2985, train MAE: 58.11 bikes/hour | val loss: 0.7573, val MAE: 88.67 bikes/hour
  ✓ Best model saved (val_mae: 88.67 bikes/hour)
Epoch 04 — train loss: 0.2436, train MAE: 52.03 bikes/hour | val loss: 0.5641, val MAE: 77.93 bikes/hour
  ✓ Best model saved (val_mae: 77.93 bikes/hour)
Epoch 05 — train loss: 0.1479, train MAE: 40.10 bikes/hour | val loss: 0.3866, val MAE: 62.32 bikes/hour
  ✓ Best model saved (val_mae: 62.32 bikes/hour)
Epoch 06 — train loss: 0.0964, train MAE: 32.66 bikes/hour | val loss: 0.3229, val MAE: 57.32 bikes/hour
  ✓ Best model saved (val_mae: 57.32 bikes/hour)
Epoch 07 — train loss: 0.0768, train MAE: 28.96 bikes/hour | val loss: 0

In [50]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("bike_lstm_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae * count_std:.2f} bikes/hour")


Test MAE: 39.12 bikes/hour


## Conclusiones

### Resultados 

| Modelo | Val MAE | Test MAE |
|---|---|---|
| Naive baseline | 93.26 bikes/h | 80.78 bikes/h |
| HistGradientBoosting | 35.02 bikes/h | 33.94 bikes/h |
| Simple RNN | 41.73 bikes/h | 40.56 bikes/h |
| GRU | 38.01 bikes/h | 36.96 bikes/h |
| LSTM | 40.43 bikes/h | 39.12 bikes/h |

### Observaciones

**1. El baseline naive fue superado fácilmente, pero el baseline sklearn no.**  
Todos los modelos RNN superaron ampliamente el baseline naive (93.26 bikes/h en validación), pero ninguno logró igualar al `HistGradientBoostingRegressor` (33.94 bikes/h en test). Esto refleja una ventaja estructural de los modelos de árboles sobre las RNNs en series temporales tabulares: los árboles reciben la información temporal a través de *lag features* construidas explícitamente, mientras que las RNNs deben aprender esa dependencia desde secuencias brutas.

**2. GRU fue el mejor modelo.**  
El GRU alcanzó un Test MAE de 36.96 bikes/h, superando tanto al Simple RNN (40.56) como al LSTM (39.12). La ventaja del GRU sobre el LSTM en este problema se explica probablemente por la menor complejidad del GRU (menos parámetros), que lo hace menos propenso a sobreajuste en un dataset de este tamaño. (~12,000 ejemplos de entrenamiento).

**3. Los tres modelos RNN quedaron dentro del rango esperado.**  
El objetivo era obtener un Test MAE inferior a 44 bikes/h (dentro de 10 bikes/h del baseline sklearn). Los tres modelos lo lograron, lo que confirma que las RNNs pueden aprender la estructura temporal del problema sin necesidad de ingeniería de features.

**4. El entrenamiento requirió muchas épocas.**  
Ningún modelo convergió en las primeras 20 épocas. El GRU necesitó 75 épocas y el LSTM 52 antes de que el early stopping se activara, lo que confirma la recomendación de usar al menos 100 épocas con early stopping (se utilizaron 200 épocas).

**5. La brecha train/val fue constante.**  
En todos los modelos, el train MAE al final del entrenamiento (16–18 bikes/h) siempre fue significativamente menor que el val MAE (38–42 bikes/h), lo que indica la presencia sobreajuste. Las técnicas aplicadas como dropout, batch normalization o reducción de `hidden_size` no redujeron esta brecha.
